# **Common Cells for Both Indian and US datasets**

In [1]:
# install everything this notebook needs
!pip install pika redis flask -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 6.6 MB/s eta 0:00:00


In [2]:
# stand up redis and rabbitmq inside this colab session
!apt-get install -y -qq redis-server rabbitmq-server > /dev/null

!redis-server --daemonize yes
!service rabbitmq-server start

[ OK ]


In [3]:
import time

# rabbitmq takes a while to fully come up, redis is basically instant
time.sleep(8)

!redis-cli ping
!rabbitmqctl status | head -3

PONG
Status of node rabbit@672afadd5313 ...
Runtime



In [ ]:
# ---- redis config ----
# same as the phase 4 notebook - replace with the real dev values once available

REDIS_HOST = 'localhost'
REDIS_PORT = 6379
REDIS_PASSWORD = None
REDIS_DB = 0
REDIS_USE_TLS = False

CACHE_TTL_SECONDS = 3600   # how long a cached prediction stays valid for

In [ ]:
# ---- rabbitmq config ----
# same as the phase 3 notebook - replace with the real dev values once available

RABBITMQ_HOST = 'localhost'
RABBITMQ_PORT = 5672
RABBITMQ_USERNAME = 'guest'
RABBITMQ_PASSWORD = 'guest'
RABBITMQ_VHOST = '/'
RABBITMQ_USE_TLS = False

QUEUE_NAME = 'yield_prediction_queue'

# **Indian Dataset**

In [4]:
# upload the pretrained model that came out of the training notebook
from google.colab import files
import pickle

print('please upload yield_model_pipeline.pkl')
uploaded_model = files.upload()

model_file = next(iter(uploaded_model))

with open(model_file, 'rb') as f:
    bundle = pickle.load(f)

model = bundle['pipeline']
num_feats = bundle['num_feats']
cat_feats = bundle['cat_feats']
required_fields = num_feats + cat_feats

print('model loaded, expecting these fields on every request:')
print(required_fields)


please upload yield_model_pipeline.pkl


Saving yield_model_pipeline.pkl to yield_model_pipeline.pkl
model loaded, expecting these fields on every request:
['urban', 'family_income', 'first_gen', 'parent_grad', 'cutoff_12th', 'entrance_score', 'tuition', 'distance_km', 'competing_offers', 'merit_aid_pct', 'need_aid_pct', 'total_aid_pct', 'aid_amount', 'net_price', 'district', 'category', 'college_tier']


In [8]:
import redis

redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=REDIS_DB,
    ssl=REDIS_USE_TLS,
    decode_responses=True
)

print('connected to redis:', redis_client.ping())

connected to redis: True


In [9]:
import pika
import ssl

# same connection helper as the phase 3 notebook. kept as a function (not one
# shared connection) because the flask thread and the consumer thread below
# both need their own connection - pika connections are not thread safe.
def get_rabbitmq_connection():

    credentials = pika.PlainCredentials(RABBITMQ_USERNAME, RABBITMQ_PASSWORD)

    if RABBITMQ_USE_TLS:
        ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        ssl_options = pika.SSLOptions(ssl_context, RABBITMQ_HOST)
        connection_params = pika.ConnectionParameters(
            host=RABBITMQ_HOST,
            port=RABBITMQ_PORT,
            virtual_host=RABBITMQ_VHOST,
            credentials=credentials,
            ssl_options=ssl_options
        )
    else:
        connection_params = pika.ConnectionParameters(
            host=RABBITMQ_HOST,
            port=RABBITMQ_PORT,
            virtual_host=RABBITMQ_VHOST,
            credentials=credentials
        )

    return pika.BlockingConnection(connection_params)


# make sure the queue exists before anything tries to publish or consume from it
setup_connection = get_rabbitmq_connection()
setup_channel = setup_connection.channel()
setup_channel.queue_declare(queue=QUEUE_NAME, durable=True)
setup_connection.close()
print('queue ready:', QUEUE_NAME)

queue ready: yield_prediction_queue


In [10]:
import json

# opens a short-lived connection, publishes one message, closes it again.
# simplest safe way to publish from inside a flask request handler without
# fighting over a connection with the background consumer thread below.
def publish_event(message):
    connection = get_rabbitmq_connection()
    channel = connection.channel()
    channel.basic_publish(
        exchange='',
        routing_key=QUEUE_NAME,
        body=json.dumps(message),
        properties=pika.BasicProperties(delivery_mode=2)
    )
    connection.close()

In [11]:
import threading
import datetime

In [12]:
# this is where "process the received message" happens on the consuming side.
# keeping the processed events in a plain list here just so we can prove, in
# this same notebook, that messages published by /predict actually get picked
# up and handled - in a real system this callback would be doing something
# like updating a dashboard, sending a notification, writing to a database etc.
processed_events = []

def handle_prediction_event(ch, method, properties, body):
    event = json.loads(body)
    event['processed_at'] = datetime.datetime.now().isoformat()
    processed_events.append(event)

    print(f"[consumer] processed prediction event for cache key {event['cache_key']} "
          f"-> yield_probability={event['result']['yield_probability']}")

    ch.basic_ack(delivery_tag=method.delivery_tag)

In [13]:
def run_consumer():
    connection = get_rabbitmq_connection()
    channel = connection.channel()
    channel.queue_declare(queue=QUEUE_NAME, durable=True)
    channel.basic_qos(prefetch_count=1)
    channel.basic_consume(queue=QUEUE_NAME, on_message_callback=handle_prediction_event)
    channel.start_consuming()


# runs forever in the background, listening for prediction events
consumer_thread = threading.Thread(target=run_consumer, daemon=True)
consumer_thread.start()

time.sleep(1)
print('consumer thread is running')

consumer thread is running


In [14]:
import hashlib
import pandas as pd
from flask import Flask, request, jsonify

In [14]:
app = Flask(__name__)

NUMERIC_FIELDS = {"urban", "family_income", "first_gen", "parent_grad", "cutoff_12th", "entrance_score",
                   "tuition", "distance_km", "competing_offers", "merit_aid_pct", "need_aid_pct",
                   "total_aid_pct", "aid_amount", "net_price"}


def validate_payload(payload):
    if not isinstance(payload, dict):
        return "request body must be a json object of applicant features"

    missing = [f for f in required_fields if f not in payload]
    if missing:
        return f"missing required fields: {missing}"

    bad_types = [f for f in NUMERIC_FIELDS if f in payload and not isinstance(payload[f], (int, float))]
    if bad_types:
        return f"these fields must be numeric: {bad_types}"

    return None


def make_cache_key(payload):
    # same applicant details should always hash to the same key, so a repeat
    # request for the exact same applicant is a cache hit
    raw = json.dumps(payload, sort_keys=True)
    return 'yield:' + hashlib.md5(raw.encode()).hexdigest()


@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'ok',
        'model_loaded': model is not None,
        'redis_connected': redis_client.ping()
    }), 200


@app.route('/predict', methods=['POST'])
def predict():
    payload = request.get_json(silent=True)

    error = validate_payload(payload)
    if error:
        return jsonify({'error': error}), 400

    cache_key = make_cache_key(payload)

    # step 2 - check redis for a cached result first
    cached = redis_client.get(cache_key)
    if cached:
        result = json.loads(cached)
        result['source'] = 'cache'
        return jsonify(result), 200

    try:
        # step 3 - not cached, so invoke the ml model
        row = pd.DataFrame([payload])[required_fields]
        proba = float(model.predict_proba(row)[:, 1][0])
        result = {
            'yield_probability': proba,
            'predicted_enrolled': proba >= 0.5
        }

        # step 4 - store the result in redis for next time
        redis_client.set(cache_key, json.dumps(result), ex=CACHE_TTL_SECONDS)

        # step 5 - publish an event to rabbitmq about this prediction
        event = {
            'event': 'prediction_made',
            'cache_key': cache_key,
            'result': result,
            'timestamp': datetime.datetime.now().isoformat()
        }
        publish_event(event)

        # step 7 - return the result. the consumer (step 6) handles the
        # published event on its own in the background thread above.
        result['source'] = 'model'
        return jsonify(result), 200

    except Exception as e:
        return jsonify({'error': str(e)}), 500


print('flask app defined: GET /health, POST /predict')

flask app defined: GET /health, POST /predict


In [15]:
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)


def is_server_already_running():
    try:
        r = requests.get('http://127.0.0.1:5000/health', timeout=1)
        return r.status_code == 200
    except Exception:
        return False


import requests

if is_server_already_running():
    print('server already running, reusing it')
else:
    flask_thread = threading.Thread(target=run_flask, daemon=True)
    flask_thread.start()
    time.sleep(2)
    print('flask server started on http://127.0.0.1:5000')

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


flask server started on http://127.0.0.1:5000


In [16]:
# health check first
r = requests.get('http://127.0.0.1:5000/health')
print(r.status_code, r.json())

INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 04:49:44] "GET /health HTTP/1.1" 200 -


200 {'model_loaded': True, 'redis_connected': True, 'status': 'ok'}


In [17]:
applicant = {
    'district': 'Madurai', 'category': 'MBC', 'urban': 0, 'family_income': 210000,
    'first_gen': 1, 'parent_grad': 0, 'cutoff_12th': 79.0, 'entrance_score': 128.0,
    'college_tier': 'Tier-2 (Affiliated)', 'tuition': 110000, 'distance_km': 35.0,
    'competing_offers': 1, 'merit_aid_pct': 0.20, 'need_aid_pct': 0.35,
    'total_aid_pct': 0.28, 'aid_amount': 30800, 'net_price': 79200,
}

# first call for this applicant - should be a cache miss, so this goes
# through the model, gets cached in redis, and publishes a rabbitmq event
print('--- call 1, expect source=model ---')
r1 = requests.post('http://127.0.0.1:5000/predict', json=applicant)
print(r1.status_code, r1.json())

# give the background consumer a moment to pick up and process the event
time.sleep(1.5)
print()
print('events processed by the consumer so far:', len(processed_events))
print(processed_events[-1] if processed_events else None)

--- call 1, expect source=model ---


INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 04:49:47] "POST /predict HTTP/1.1" 200 -


[consumer] processed prediction event for cache key yield:07f0dfa67846f10fbb5bed79aadec3dc -> yield_probability=0.3799366710209084
200 {'predicted_enrolled': False, 'source': 'model', 'yield_probability': 0.3799366710209084}

events processed by the consumer so far: 1
{'event': 'prediction_made', 'cache_key': 'yield:07f0dfa67846f10fbb5bed79aadec3dc', 'result': {'yield_probability': 0.3799366710209084, 'predicted_enrolled': False}, 'timestamp': '2026-09-25T04:49:46.136925', 'processed_at': '2026-09-25T04:49:46.970846'}


In [18]:
# second call, same applicant - should now be a cache hit, no model call,
# no new rabbitmq event
print('--- call 2, expect source=cache ---')
r2 = requests.post('http://127.0.0.1:5000/predict', json=applicant)
print(r2.status_code, r2.json())

print()
print('events processed by the consumer (should be unchanged):', len(processed_events))

INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 04:49:51] "POST /predict HTTP/1.1" 200 -


--- call 2, expect source=cache ---
200 {'predicted_enrolled': False, 'source': 'cache', 'yield_probability': 0.3799366710209084}

events processed by the consumer (should be unchanged): 1


In [19]:
# manually delete the cached entry, same as the update/delete demo in the
# phase 4 notebook, then call again to confirm it goes back through the
# model and gets re-cached
cache_key = make_cache_key(applicant)
redis_client.delete(cache_key)
print('deleted cache key:', cache_key)

print()
print('--- call 3 after delete, expect source=model again ---')
r3 = requests.post('http://127.0.0.1:5000/predict', json=applicant)
print(r3.status_code, r3.json())

time.sleep(1.5)
print()
print('total events processed by the consumer:', len(processed_events))

deleted cache key: yield:07f0dfa67846f10fbb5bed79aadec3dc

--- call 3 after delete, expect source=model again ---


INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 04:49:54] "POST /predict HTTP/1.1" 200 -


[consumer] processed prediction event for cache key yield:07f0dfa67846f10fbb5bed79aadec3dc -> yield_probability=0.37993667102090845
200 {'predicted_enrolled': False, 'source': 'model', 'yield_probability': 0.37993667102090845}

total events processed by the consumer: 2


In [20]:
# one more check with a validation error - missing fields should come back
# as a 400, not a 500 or a silent wrong prediction
incomplete_applicant = {'district': 'Chennai', 'family_income': 300000}
r4 = requests.post('http://127.0.0.1:5000/predict', json=incomplete_applicant)
print(r4.status_code, r4.json())

INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 04:49:59] "POST /predict HTTP/1.1" 400 -


400 {'error': "missing required fields: ['urban', 'first_gen', 'parent_grad', 'cutoff_12th', 'entrance_score', 'tuition', 'distance_km', 'competing_offers', 'merit_aid_pct', 'need_aid_pct', 'total_aid_pct', 'aid_amount', 'net_price', 'category', 'college_tier']"}


In [21]:
applicants = [
    {
        "district": "Madurai",
        "category": "MBC",
        "urban": 0,
        "family_income": 210000,
        "first_gen": 1,
        "parent_grad": 0,
        "cutoff_12th": 79.0,
        "entrance_score": 128.0,
        "college_tier": "Tier-2 (Affiliated)",
        "tuition": 110000,
        "distance_km": 35.0,
        "competing_offers": 1,
        "merit_aid_pct": 0.2,
        "need_aid_pct": 0.35,
        "total_aid_pct": 0.28,
        "aid_amount": 30800,
        "net_price": 79200
    },
    {
        "district": "Coimbatore",
        "category": "General",
        "urban": 1,
        "family_income": 500000,
        "first_gen": 1,
        "parent_grad": 0,
        "cutoff_12th": 90.0,
        "entrance_score": 80.0,
        "college_tier": "Tier-1",
        "tuition": 250000,
        "distance_km": 25.0,
        "competing_offers": 2,
        "merit_aid_pct": 0.2,
        "need_aid_pct": 0.1,
        "total_aid_pct": 0.3,
        "aid_amount": 45000,
        "net_price": 150000
    },
    {
        "district": "Chennai",
        "category": "BC",
        "urban": 1,
        "family_income": 350000,
        "first_gen": 0,
        "parent_grad": 1,
        "cutoff_12th": 86.0,
        "entrance_score": 92.0,
        "college_tier": "Tier-2 (Affiliated)",
        "tuition": 160000,
        "distance_km": 18.0,
        "competing_offers": 3,
        "merit_aid_pct": 0.15,
        "need_aid_pct": 0.2,
        "total_aid_pct": 0.25,
        "aid_amount": 40000,
        "net_price": 120000
    }
]

for i, applicant in enumerate(applicants, start=1):

    print("\n" + "=" * 70)
    print(f"APPLICANT {i}")
    print("=" * 70)

    # 1. send applicant to Flask /predict
    response = requests.post('http://127.0.0.1:5000/predict', json=applicant)
    print("\n1. API OUTPUT")
    print("Status Code:", response.status_code)
    print(json.dumps(response.json(), indent=2))

    # 2. generate the exact same redis cache key client-side
    cache_key = make_cache_key(applicant)
    print("\n2. REDIS CACHE KEY")
    print(cache_key)

    # 3. fetch only this key from redis
    stored_data = redis_client.get(cache_key)
    print("\n3. DATA FETCHED FROM REDIS")
    if stored_data is None:
        print("No data found in Redis for this key.")
    else:
        print(json.dumps(json.loads(stored_data), indent=2))

    # 4. remaining cache time
    ttl = redis_client.ttl(cache_key)
    print("\n4. REDIS TTL")
    print(ttl, "seconds")

    # 5. confirm the key exists
    print("\n5. REDIS KEY EXISTS")
    print(redis_client.exists(cache_key))


INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 04:50:01] "POST /predict HTTP/1.1" 200 -



APPLICANT 1

1. API OUTPUT
Status Code: 200
{
  "predicted_enrolled": false,
  "source": "cache",
  "yield_probability": 0.37993667102090845
}

2. REDIS CACHE KEY
yield:07f0dfa67846f10fbb5bed79aadec3dc

3. DATA FETCHED FROM REDIS
{
  "yield_probability": 0.37993667102090845,
  "predicted_enrolled": false
}

4. REDIS TTL
3592 seconds

5. REDIS KEY EXISTS
1

APPLICANT 2


INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 04:50:03] "POST /predict HTTP/1.1" 200 -


[consumer] processed prediction event for cache key yield:04e5cb93d7f7f7415bb2969fd6b1ded0 -> yield_probability=0.5723586576755085

1. API OUTPUT
Status Code: 200
{
  "predicted_enrolled": true,
  "source": "model",
  "yield_probability": 0.5723586576755085
}

2. REDIS CACHE KEY
yield:04e5cb93d7f7f7415bb2969fd6b1ded0

3. DATA FETCHED FROM REDIS
{
  "yield_probability": 0.5723586576755085,
  "predicted_enrolled": true
}

4. REDIS TTL
3599 seconds

5. REDIS KEY EXISTS
1

APPLICANT 3


INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 04:50:04] "POST /predict HTTP/1.1" 200 -


[consumer] processed prediction event for cache key yield:5ee585b4c80b9a4701b4ff594d547775 -> yield_probability=0.5549602509416233

1. API OUTPUT
Status Code: 200
{
  "predicted_enrolled": true,
  "source": "model",
  "yield_probability": 0.5549602509416233
}

2. REDIS CACHE KEY
yield:5ee585b4c80b9a4701b4ff594d547775

3. DATA FETCHED FROM REDIS
{
  "yield_probability": 0.5549602509416233,
  "predicted_enrolled": true
}

4. REDIS TTL
3599 seconds

5. REDIS KEY EXISTS
1


# **US Dataset**

In [6]:
# upload the pretrained US yield model
from google.colab import files
import pickle

print('please upload us_yield_model_pipeline.pkl')
uploaded_model = files.upload()

model_file = next(iter(uploaded_model))

with open(model_file, 'rb') as f:
    model = pickle.load(f)

# The uploaded .pkl contains the Pipeline directly.
# Extract the feature columns from the pipeline's preprocessor.
preprocessor = model.named_steps['preprocessor']

num_feats = []
cat_feats = []

for name, transformer, columns in preprocessor.transformers_:
    if name == 'num':
        num_feats = list(columns)
    elif name == 'cat':
        cat_feats = list(columns)

required_fields = num_feats + cat_feats

print('model loaded successfully')
print('model type:', type(model))
print('numeric fields:', num_feats)
print('categorical fields:', cat_feats)
print('required fields:', required_fields)

please upload us_yield_model_pipeline.pkl


Saving yield_model_pipeline.pkl to yield_model_pipeline.pkl


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


model loaded successfully
model type: <class 'sklearn.pipeline.Pipeline'>
numeric fields: ['is_in_state', 'student_aid_index', 'adjusted_gross_income', 'first_gen', 'pell_eligible', 'hs_gpa', 'sat_act_percentile', 'institutional_tier', 'cost_of_attendance', 'miles_from_campus', 'fafsa_month_sin', 'fafsa_month_cos', 'demonstrated_interest', 'merit_scholarship_amt', 'need_grant_amt', 'net_price', 'net_price_to_income_ratio', 'financial_aid_discount_rate', 'unmet_financial_need_gap', 'engagement_velocity', 'urban_centric_locale_Rural', 'urban_centric_locale_Suburb', 'urban_centric_locale_Town']
categorical fields: []
required fields: ['is_in_state', 'student_aid_index', 'adjusted_gross_income', 'first_gen', 'pell_eligible', 'hs_gpa', 'sat_act_percentile', 'institutional_tier', 'cost_of_attendance', 'miles_from_campus', 'fafsa_month_sin', 'fafsa_month_cos', 'demonstrated_interest', 'merit_scholarship_amt', 'need_grant_amt', 'net_price', 'net_price_to_income_ratio', 'financial_aid_discount_

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
import redis

redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=REDIS_DB,
    ssl=REDIS_USE_TLS,
    decode_responses=True
)

print('connected to redis:', redis_client.ping())


connected to redis: True


In [8]:
import pika
import ssl

# kept as a function (not one shared connection) because the flask thread and the
# consumer thread below both need their own connection - pika connections are not
# thread safe.
def get_rabbitmq_connection():

    credentials = pika.PlainCredentials(RABBITMQ_USERNAME, RABBITMQ_PASSWORD)

    if RABBITMQ_USE_TLS:
        ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        ssl_options = pika.SSLOptions(ssl_context, RABBITMQ_HOST)
        connection_params = pika.ConnectionParameters(
            host=RABBITMQ_HOST,
            port=RABBITMQ_PORT,
            virtual_host=RABBITMQ_VHOST,
            credentials=credentials,
            ssl_options=ssl_options
        )
    else:
        connection_params = pika.ConnectionParameters(
            host=RABBITMQ_HOST,
            port=RABBITMQ_PORT,
            virtual_host=RABBITMQ_VHOST,
            credentials=credentials
        )

    return pika.BlockingConnection(connection_params)


# make sure the queue exists before anything tries to publish or consume from it
setup_connection = get_rabbitmq_connection()
setup_channel = setup_connection.channel()
setup_channel.queue_declare(queue=QUEUE_NAME, durable=True)
setup_connection.close()
print('queue ready:', QUEUE_NAME)

queue ready: yield_prediction_queue


In [9]:
import json

# opens a short-lived connection, publishes one message, closes it again.
# simplest safe way to publish from inside a flask request handler without
# fighting over a connection with the background consumer thread below.
def publish_event(message):
    connection = get_rabbitmq_connection()
    channel = connection.channel()
    channel.basic_publish(
        exchange='',
        routing_key=QUEUE_NAME,
        body=json.dumps(message),
        properties=pika.BasicProperties(delivery_mode=2)
    )
    connection.close()

In [10]:
import threading
import datetime

# keeping the processed events in a plain list here just so we can prove, in this
# same notebook, that messages published by /predict actually get picked up and
# handled - in a real system this callback would be doing something like updating a
# dashboard, sending a notification, writing to a database etc.
processed_events = []

def handle_prediction_event(ch, method, properties, body):
    event = json.loads(body)
    event['processed_at'] = datetime.datetime.now().isoformat()
    processed_events.append(event)

    print(f"[consumer] processed prediction event for cache key {event['cache_key']} "
          f"-> yield_probability={event['result']['yield_probability']}")

    ch.basic_ack(delivery_tag=method.delivery_tag)

In [11]:
def run_consumer():
    connection = get_rabbitmq_connection()
    channel = connection.channel()
    channel.queue_declare(queue=QUEUE_NAME, durable=True)
    channel.basic_qos(prefetch_count=1)
    channel.basic_consume(queue=QUEUE_NAME, on_message_callback=handle_prediction_event)
    channel.start_consuming()


# runs forever in the background, listening for prediction events
consumer_thread = threading.Thread(target=run_consumer, daemon=True)
consumer_thread.start()

time.sleep(1)
print('consumer thread is running')

consumer thread is running


In [12]:
import hashlib
import pandas as pd
from flask import Flask, request, jsonify

app = Flask(__name__)

# FIX: derive from the bundle instead of hardcoding a dataset-specific field list —
# this makes the validation logic correct for whichever schema this notebook serves.
NUMERIC_FIELDS = set(num_feats)


def validate_payload(payload):
    if not isinstance(payload, dict):
        return "request body must be a json object of applicant features"

    missing = [f for f in required_fields if f not in payload]
    if missing:
        return f"missing required fields: {missing}"

    bad_types = [f for f in NUMERIC_FIELDS if f in payload and not isinstance(payload[f], (int, float))]
    if bad_types:
        return f"these fields must be numeric: {bad_types}"

    return None


def make_cache_key(payload):
    # same applicant details should always hash to the same key, so a repeat
    # request for the exact same applicant is a cache hit
    raw = json.dumps(payload, sort_keys=True)
    return 'yield:' + hashlib.md5(raw.encode()).hexdigest()


@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status': 'ok',
        'model_loaded': model is not None,
        'redis_connected': redis_client.ping()
    }), 200


@app.route('/predict', methods=['POST'])
def predict():
    payload = request.get_json(silent=True)

    error = validate_payload(payload)
    if error:
        return jsonify({'error': error}), 400

    cache_key = make_cache_key(payload)

    # step 2 - check redis for a cached result first
    cached = redis_client.get(cache_key)
    if cached:
        result = json.loads(cached)
        result['source'] = 'cache'
        return jsonify(result), 200

    try:
        # step 3 - not cached, so invoke the ml model
        row = pd.DataFrame([payload])[required_fields]
        proba = float(model.predict_proba(row)[:, 1][0])
        result = {
            'yield_probability': proba,
            'predicted_enrolled': proba >= 0.5
        }

        # step 4 - store the result in redis for next time
        redis_client.set(cache_key, json.dumps(result), ex=CACHE_TTL_SECONDS)

        # step 5 - publish an event to rabbitmq about this prediction
        event = {
            'event': 'prediction_made',
            'cache_key': cache_key,
            'result': result,
            'timestamp': datetime.datetime.now().isoformat()
        }
        publish_event(event)

        # step 7 - return the result. the consumer (step 6) handles the
        # published event on its own in the background thread above.
        result['source'] = 'model'
        return jsonify(result), 200

    except Exception as e:
        return jsonify({'error': str(e)}), 500


print('flask app defined: GET /health, POST /predict')

flask app defined: GET /health, POST /predict


In [13]:
import requests

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)


def is_server_already_running():
    try:
        r = requests.get('http://127.0.0.1:5000/health', timeout=1)
        return r.status_code == 200
    except Exception:
        return False


if is_server_already_running():
    print('server already running, reusing it')
else:
    flask_thread = threading.Thread(target=run_flask, daemon=True)
    flask_thread.start()
    time.sleep(2)
    print('flask server started on http://127.0.0.1:5000')

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


flask server started on http://127.0.0.1:5000


In [14]:
# health check first
r = requests.get('http://127.0.0.1:5000/health')
print(r.status_code, r.json())

INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 05:21:15] "GET /health HTTP/1.1" 200 -


200 {'model_loaded': True, 'redis_connected': True, 'status': 'ok'}


In [15]:
applicant = {
    "is_in_state": 0,
    "student_aid_index": 47307.0,
    "adjusted_gross_income": 206264.0,
    "first_gen": 0,
    "pell_eligible": 0,
    "hs_gpa": 3.61,
    "sat_act_percentile": 70.0,
    "institutional_tier": 1,
    "cost_of_attendance": 50454.0,
    "miles_from_campus": 169.8,
    "fafsa_month_sin": 0.5,
    "fafsa_month_cos": -0.866,
    "demonstrated_interest": 40.0,
    "merit_scholarship_amt": 22292.0,
    "need_grant_amt": 820.0,
    "net_price": 27342.0,
    "net_price_to_income_ratio": 0.124516,
    "financial_aid_discount_rate": 0.4581,
    "unmet_financial_need_gap": 0.0,
    "engagement_velocity": 0.2286,
    "urban_centric_locale_Rural": 0,
    "urban_centric_locale_Suburb": 1,
    "urban_centric_locale_Town": 0
}

# first call for this applicant - should be a cache miss, so this goes
# through the model, gets cached in redis, and publishes a rabbitmq event
print('--- call 1, expect source=model ---')
r1 = requests.post('http://127.0.0.1:5000/predict', json=applicant)
print(r1.status_code, r1.json())

# give the background consumer a moment to pick up and process the event
time.sleep(1.5)
print()
print('events processed by the consumer so far:', len(processed_events))
print(processed_events[-1] if processed_events else None)

--- call 1, expect source=model ---


INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 05:21:20] "POST /predict HTTP/1.1" 200 -


[consumer] processed prediction event for cache key yield:a10e230e2590e16a015cea72061f9b8f -> yield_probability=0.7656958699226379
200 {'predicted_enrolled': True, 'source': 'model', 'yield_probability': 0.7656958699226379}

events processed by the consumer so far: 1
{'event': 'prediction_made', 'cache_key': 'yield:a10e230e2590e16a015cea72061f9b8f', 'result': {'yield_probability': 0.7656958699226379, 'predicted_enrolled': True}, 'timestamp': '2026-09-25T05:21:20.466804', 'processed_at': '2026-09-25T05:21:20.643559'}


In [16]:
# second call, same applicant - should now be a cache hit, no model call,
# no new rabbitmq event
print('--- call 2, expect source=cache ---')
r2 = requests.post('http://127.0.0.1:5000/predict', json=applicant)
print(r2.status_code, r2.json())

print()
print('events processed by the consumer (should be unchanged):', len(processed_events))

INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 05:21:28] "POST /predict HTTP/1.1" 200 -


--- call 2, expect source=cache ---
200 {'predicted_enrolled': True, 'source': 'cache', 'yield_probability': 0.7656958699226379}

events processed by the consumer (should be unchanged): 1


In [17]:
# manually delete the cached entry, then call again to confirm it goes back
# through the model and gets re-cached
cache_key = make_cache_key(applicant)
redis_client.delete(cache_key)
print('deleted cache key:', cache_key)

print()
print('--- call 3 after delete, expect source=model again ---')
r3 = requests.post('http://127.0.0.1:5000/predict', json=applicant)
print(r3.status_code, r3.json())

time.sleep(1.5)
print()
print('total events processed by the consumer:', len(processed_events))

deleted cache key: yield:a10e230e2590e16a015cea72061f9b8f

--- call 3 after delete, expect source=model again ---


INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 05:21:29] "POST /predict HTTP/1.1" 200 -


[consumer] processed prediction event for cache key yield:a10e230e2590e16a015cea72061f9b8f -> yield_probability=0.7656958699226379
200 {'predicted_enrolled': True, 'source': 'model', 'yield_probability': 0.7656958699226379}

total events processed by the consumer: 2


In [18]:
# validation error check - missing fields should come back as a 400,
# not a 500 or a silent wrong prediction
incomplete_applicant = {"adjusted_gross_income": 60000.0, "hs_gpa": 3.4}
r4 = requests.post('http://127.0.0.1:5000/predict', json=incomplete_applicant)
print(r4.status_code, r4.json())

INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 05:21:34] "POST /predict HTTP/1.1" 400 -


400 {'error': "missing required fields: ['is_in_state', 'student_aid_index', 'first_gen', 'pell_eligible', 'sat_act_percentile', 'institutional_tier', 'cost_of_attendance', 'miles_from_campus', 'fafsa_month_sin', 'fafsa_month_cos', 'demonstrated_interest', 'merit_scholarship_amt', 'need_grant_amt', 'net_price', 'net_price_to_income_ratio', 'financial_aid_discount_rate', 'unmet_financial_need_gap', 'engagement_velocity', 'urban_centric_locale_Rural', 'urban_centric_locale_Suburb', 'urban_centric_locale_Town']"}


In [19]:
applicants = [
    {
        "is_in_state": 0,
        "student_aid_index": 47307.0,
        "adjusted_gross_income": 206264.0,
        "first_gen": 0,
        "pell_eligible": 0,
        "hs_gpa": 3.61,
        "sat_act_percentile": 70.0,
        "institutional_tier": 1,
        "cost_of_attendance": 50454.0,
        "miles_from_campus": 169.8,
        "fafsa_month_sin": 0.5,
        "fafsa_month_cos": -0.866,
        "demonstrated_interest": 40.0,
        "merit_scholarship_amt": 22292.0,
        "need_grant_amt": 820.0,
        "net_price": 27342.0,
        "net_price_to_income_ratio": 0.124516,
        "financial_aid_discount_rate": 0.4581,
        "unmet_financial_need_gap": 0.0,
        "engagement_velocity": 0.2286,
        "urban_centric_locale_Rural": 0,
        "urban_centric_locale_Suburb": 1,
        "urban_centric_locale_Town": 0
    },
    {
        "is_in_state": 1,
        "student_aid_index": 3408.0,
        "adjusted_gross_income": 28634.0,
        "first_gen": 0,
        "pell_eligible": 1,
        "hs_gpa": 3.82,
        "sat_act_percentile": 93.0,
        "institutional_tier": 0,
        "cost_of_attendance": 18794.0,
        "miles_from_campus": 10.3,
        "fafsa_month_sin": -1.0,
        "fafsa_month_cos": -0.0,
        "demonstrated_interest": 48.4,
        "merit_scholarship_amt": 1404.0,
        "need_grant_amt": 8103.0,
        "net_price": 9287.0,
        "net_price_to_income_ratio": 0.280884,
        "financial_aid_discount_rate": 0.5059,
        "unmet_financial_need_gap": 5879.0,
        "engagement_velocity": 0.047,
        "urban_centric_locale_Rural": 0,
        "urban_centric_locale_Suburb": 0,
        "urban_centric_locale_Town": 0
    },
    {
        "is_in_state": 0,
        "student_aid_index": 4787.0,
        "adjusted_gross_income": 35389.0,
        "first_gen": 0,
        "pell_eligible": 1,
        "hs_gpa": 3.14,
        "sat_act_percentile": 65.0,
        "institutional_tier": 3,
        "cost_of_attendance": 84829.0,
        "miles_from_campus": 82.8,
        "fafsa_month_sin": -0.866,
        "fafsa_month_cos": -0.5,
        "demonstrated_interest": 20.8,
        "merit_scholarship_amt": 23787.0,
        "need_grant_amt": 33635.0,
        "net_price": 27407.0,
        "net_price_to_income_ratio": 0.573462,
        "financial_aid_discount_rate": 0.6769,
        "unmet_financial_need_gap": 22620.0,
        "engagement_velocity": 0.0079,
        "urban_centric_locale_Rural": 0,
        "urban_centric_locale_Suburb": 1,
        "urban_centric_locale_Town": 0
    }
]

for i, applicant in enumerate(applicants, start=1):

    print("\n" + "=" * 70)
    print(f"APPLICANT {i}")
    print("=" * 70)

    # 1. send applicant to Flask /predict
    response = requests.post('http://127.0.0.1:5000/predict', json=applicant)
    print("\n1. API OUTPUT")
    print("Status Code:", response.status_code)
    print(json.dumps(response.json(), indent=2))

    # 2. generate the exact same redis cache key client-side
    cache_key = make_cache_key(applicant)
    print("\n2. REDIS CACHE KEY")
    print(cache_key)

    # 3. fetch only this key from redis
    stored_data = redis_client.get(cache_key)
    print("\n3. DATA FETCHED FROM REDIS")
    if stored_data is None:
        print("No data found in Redis for this key.")
    else:
        print(json.dumps(json.loads(stored_data), indent=2))

    # 4. remaining cache time
    ttl = redis_client.ttl(cache_key)
    print("\n4. REDIS TTL")
    print(ttl, "seconds")

    # 5. confirm the key exists
    print("\n5. REDIS KEY EXISTS")
    print(redis_client.exists(cache_key))

INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 05:21:37] "POST /predict HTTP/1.1" 200 -



APPLICANT 1

1. API OUTPUT
Status Code: 200
{
  "predicted_enrolled": true,
  "source": "cache",
  "yield_probability": 0.7656958699226379
}

2. REDIS CACHE KEY
yield:a10e230e2590e16a015cea72061f9b8f

3. DATA FETCHED FROM REDIS
{
  "yield_probability": 0.7656958699226379,
  "predicted_enrolled": true
}

4. REDIS TTL
3592 seconds

5. REDIS KEY EXISTS
1

APPLICANT 2


INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 05:21:38] "POST /predict HTTP/1.1" 200 -


[consumer] processed prediction event for cache key yield:198040a58f734a6339a8b1faf21dbe75 -> yield_probability=0.5854235887527466

1. API OUTPUT
Status Code: 200
{
  "predicted_enrolled": true,
  "source": "model",
  "yield_probability": 0.5854235887527466
}

2. REDIS CACHE KEY
yield:198040a58f734a6339a8b1faf21dbe75

3. DATA FETCHED FROM REDIS
{
  "yield_probability": 0.5854235887527466,
  "predicted_enrolled": true
}

4. REDIS TTL
3600 seconds

5. REDIS KEY EXISTS
1

APPLICANT 3


INFO:werkzeug:127.0.0.1 - - [25/Sep/2026 05:21:38] "POST /predict HTTP/1.1" 200 -


[consumer] processed prediction event for cache key yield:e93e090716d597f512c1213045c04e80 -> yield_probability=0.5896974802017212

1. API OUTPUT
Status Code: 200
{
  "predicted_enrolled": true,
  "source": "model",
  "yield_probability": 0.5896974802017212
}

2. REDIS CACHE KEY
yield:e93e090716d597f512c1213045c04e80

3. DATA FETCHED FROM REDIS
{
  "yield_probability": 0.5896974802017212,
  "predicted_enrolled": true
}

4. REDIS TTL
3600 seconds

5. REDIS KEY EXISTS
1
